In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pytorch_lightning as pl
import yaml
import argparse
import torch
import h5py
import matplotlib.pyplot as plt
import numpy as np
#import openslide
import pandas as pd
import matplotlib.patches as patches
from tqdm import tqdm
import glob
import torchmetrics
import os
from matplotlib.colors import LinearSegmentedColormap

from classifier import ClassifierLightning
from options import Options
from mpl_toolkits.axes_grid1 import make_axes_locatable


/lambda_stor/homes/pvasanthakumari/miniconda/envs/wsi/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using PPEG positional encoding


In [2]:
# function that plots scores nicely. scores should have the same length as the number of tiles.
def NormalizeData(data):
    return (data - data.min()) / (data.max() - data.min())

def plot_scores(coords, scores, image, overlay=True, clamp=0.05, norm=True, colormap='RdBu', crop=False, indices=[]):
    if clamp:
        q05, q95 = torch.quantile(scores, clamp), torch.quantile(scores, 1-clamp)
        scores.clamp_(q05,q95)
    
    if norm:
        scores = NormalizeData(scores)
        
    if crop:
        coords_min, coords_max = np.array(coords).min(axis=0), np.array(coords).max(axis=0)
        y_min, y_max, x_min, x_max = round(coords_min[1]/d), round(coords_max[1]/d), round(coords_min[0]/d), round(coords_max[0]/d)
        if slide_path.stem == '439042':
            x_max = round((69 * 1013)/d)
        print(y_min, y_max, x_min, x_max)
    else:
        y_min, y_max, x_min, x_max = 0, image.shape[0], 0, image.shape[1]
        
        
    attention_map = np.zeros((image.shape[0], image.shape[1]), dtype=np.float32)    
    tissue_map = -np.ones((image.shape[0], image.shape[1]), dtype=np.float32)
    
    offset = 1013
    for (x,y), s in zip(coords, scores):
        
        if colormap == 'RdBu': 
            attention_map[round(y/d):round((y+offset)/d), round(x/d):round((x+offset)/d)] = 1 - s.item()
        else: 
            attention_map[round(y/d):round((y+offset)/d), round(x/d):round((x+offset)/d)] = s.item()
        tissue_map[round(y/d):round((y+offset)/d), round(x/d):round((x+offset)/d)] = s.item()
       
    attention_map = np.array(attention_map * 255., dtype=np.uint8)
    tissue_map[tissue_map>=0] = 1
    tissue_map[tissue_map<0] = 0

    if len(indices) != 0:
        highlight_map = np.zeros((image.shape[0], image.shape[1]), dtype=np.float32)    
        for i in indices:
            x, y = coords[i]
            highlight_map[round(y/d):round((y+offset)/d), round(x/d):round((x+offset)/d)] = 1    
                 
#     plt.figure(figsize=(30, 30))
    a = 1.
    if overlay:
        plt.imshow(image[y_min:y_max, x_min:x_max])
        a = 0.5
    
    if crop:
        plt.imshow(attention_map[y_min:y_max, x_min:x_max], alpha=a*(tissue_map[y_min:y_max, x_min:x_max]), cmap=colormap, interpolation='nearest')
#         plt.imshow(attention_map[round(coords_min[1]/d):, round(coords_min[0]/d):], alpha=a*(tissue_map[round(coords_min[1]/d):, round(coords_min[0]/d):]), cmap=colormap, interpolation='nearest')
    else:
        plt.imshow(attention_map, alpha=a*(tissue_map), cmap=colormap, interpolation='nearest')
    
    if len(indices) != 0:
        plt.imshow(highlight_map[y_min:y_max, x_min:x_max], alpha=1.*(highlight_map), cmap='viridis', interpolation='nearest')
    
    plt.axis('off')

In [3]:
def compute_rollout_attention(all_layer_matrices, start_layer=0):
    # adding residual consideration- code adapted from https://github.com/samiraabnar/attention_flow
    num_tokens = all_layer_matrices[0].shape[1]
    batch_size = all_layer_matrices[0].shape[0]
    eye = torch.eye(num_tokens).expand(batch_size, num_tokens, num_tokens).to(all_layer_matrices[0].device)
    all_layer_matrices = [all_layer_matrices[i] + eye for i in range(len(all_layer_matrices))]
    matrices_aug = [all_layer_matrices[i] / all_layer_matrices[i].sum(dim=-1, keepdim=True)
                          for i in range(len(all_layer_matrices))]
    joint_attention = matrices_aug[start_layer]
    for i in range(start_layer+1, len(matrices_aug)):
        joint_attention = matrices_aug[i].bmm(joint_attention)
    return joint_attention

In [4]:
def generate_rollout(model, input, start_layer=0):
    model(input)
    blocks = model.transformer.layers
    all_layer_attentions = []
    for blk in blocks:
        attn_heads = blk[0].fn.get_attention_map()
        avg_heads = (attn_heads.sum(dim=1) / attn_heads.shape[1]).detach()
        all_layer_attentions.append(avg_heads)
    rollout = compute_rollout_attention(all_layer_attentions, start_layer=start_layer)
    return rollout[:,0, 1:]

### We need to figure out the pixel positions of the generated roll-outs. For this, the csv files that are input to CTransPath contain the order in which the features are generated for the image patches 

In [5]:
path = '/nfs/ml_lab/projects/Pilot1_PreclinicalHPC/mayo_data/All_patches/'
pixels_features={}
for file in os.listdir(path):
    if file.endswith('.csv'):
        pixels=[]
        filename=file[:-4]
        file_read = pd.read_csv(str(path+file),header=None)
        for i in range(len(file_read)):
            pixels.append((int(file_read[0][i].split('_')[-3]), int(file_read[0][i].split('_')[-2])))
        pixels_features[filename] = pixels

In [6]:
config_file='config.yaml'
# Load the configuration from the YAML file
with open(config_file, 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

print('\n--- load options ---')
for name, value in sorted(config.items()):
    print(f'{name}: {str(value)}')

cfg = argparse.Namespace(**config)


--- load options ---
bs: 1
clini_info: {}
cohorts: ['CRC']
criterion: BCEWithLogitsLoss
data_config: data_config.yaml
ext_cohorts: ['CRC']
feats: ctranspath
folds: 5
input_dim: 768
label_dict: {'MSS': 0, 'MSI': 1}
lr: 2e-05
lr_scheduler: None
lr_scheduler_config: None
model: Transformer
model_config: {'heads': 8, 'dim_head': 64, 'dim': 512, 'mlp_dim': 512}
name: debug
norm: raw
num_classes: 1
num_epochs: 8
num_samples: None
num_tiles: -1
optimizer: AdamW
pad_tiles: False
project: MSI
save_dir: logs
seed: None
stop_criterion: loss
target: TARGET
task: binary
val_check_interval: 500
wd: 2e-05


In [18]:
experiment = 'Test_Aug19_Transformer_CRC_raw_TARGET'
model_path = str('/nfs/ml_lab/projects/Pilot1_PreclinicalHPC/mayo_data/HistoBistro/logs/' + experiment + '/models/')
folds = 5
model_folds={}
for fold in range(folds):
    model_folds[fold] = str(model_path + 'best_model_'+experiment+'_fold'+str(fold)+'.ckpt')

feature_path = '/nfs/ml_lab/projects/Pilot1_PreclinicalHPC/mayo_data/HistoBistro/h5_embeddings/'
files = glob.glob(str(feature_path + '/*.h5'))
file_folds={}
for fold in range(folds):
    file_folds[fold] = []
    patient_fold = pd.read_csv(str(feature_path + 'folds/TARGET_5folds/fold'+str(fold)+'_test.csv'))['PATIENT']
    for file in files:
        patient=file.split('/')[-1].split('_')[0]+'_'+file.split('/')[-1].split('_')[1]+'_'+file.split('/')[-1].split('_')[2]
        if patient in patient_fold.values:   
            file_folds[fold].append(file)


Mayo_MSI_0019
Mayo_MSS_0005
Mayo_MSS_0012
Mayo_MSI_0020
Mayo_MSS_0013
Mayo_MSS_0004
Mayo_MSI_0018
Mayo_MSS_0006
Mayo_MSS_0011
Mayo_MSS_0010
Mayo_MSS_0007
Mayo_MSI_0008
Mayo_MSS_0014
Mayo_MSS_0003
Mayo_MSS_0002
Mayo_MSS_0015
Mayo_MSI_0009
Mayo_MSS_0017
Mayo_MSS_0001
Mayo_MSS_0016
Mayo_MSS_0008
Mayo_MSI_0014
Mayo_MSI_0003
Mayo_MSI_0002
Mayo_MSI_0015
Mayo_MSS_0009
Mayo_MSI_0017
Mayo_MSI_0001
Mayo_MSI_0016
Mayo_MSS_0019
Mayo_MSI_0005
Mayo_MSI_0012
Mayo_MSI_0013
Mayo_MSS_0020
Mayo_MSI_0004
Mayo_MSS_0018
Mayo_MSI_0006
Mayo_MSI_0011
Mayo_MSI_0010
Mayo_MSI_0007
Mayo_MSI_0019
Mayo_MSS_0005
Mayo_MSS_0012
Mayo_MSI_0020
Mayo_MSS_0013
Mayo_MSS_0004
Mayo_MSI_0018
Mayo_MSS_0006
Mayo_MSS_0011
Mayo_MSS_0010
Mayo_MSS_0007
Mayo_MSI_0008
Mayo_MSS_0014
Mayo_MSS_0003
Mayo_MSS_0002
Mayo_MSS_0015
Mayo_MSI_0009
Mayo_MSS_0017
Mayo_MSS_0001
Mayo_MSS_0016
Mayo_MSS_0008
Mayo_MSI_0014
Mayo_MSI_0003
Mayo_MSI_0002
Mayo_MSI_0015
Mayo_MSS_0009
Mayo_MSI_0017
Mayo_MSI_0001
Mayo_MSI_0016
Mayo_MSS_0019
Mayo_MSI_0005
Mayo_M

In [20]:
name  = 'debug'
target = 'TARGET'
folds = 5
path = '/nfs/ml_lab/projects/Pilot1_PreclinicalHPC/mayo_data/HistoBistro/h5_embeddings'
files = glob.glob(str(path + '/*.h5'))
#print(files)
rollout={}
file_names=[]
features_all ={}
for fold in range(folds):
    # Model
    model_path = Path(model_folds[fold])
    cfg.pos_weight = torch.tensor([1.0])
    classifier = ClassifierLightning(cfg)
    checkpoint = torch.load(model_path, map_location=lambda storage, loc: storage)
    checkpoint['state_dict'].keys()
    classifier.load_state_dict(checkpoint['state_dict'])
    classifier.eval();
    # Get roll out
    for file in file_folds[fold]:
        file_name = file.split('/')[-1].split('.')[0]
        file_names.append(file_name)
        h5_file = h5py.File(file)
        features = torch.Tensor(np.array(h5_file['features'])).unsqueeze(0)
        features_all[file_name] = features
        rollout[file_name] = generate_rollout(classifier.model, features, start_layer=0).squeeze(0)
print(file_names)


['Mayo_MSI_0018', 'Mayo_MSS_0010', 'Mayo_MSI_0008', 'Mayo_MSS_0003', 'Mayo_MSS_0001', 'Mayo_MSI_0001', 'Mayo_MSS_0020', 'Mayo_MSI_0011', 'Mayo_MSS_0004', 'Mayo_MSS_0011', 'Mayo_MSS_0007', 'Mayo_MSS_0015', 'Mayo_MSI_0003', 'Mayo_MSI_0013', 'Mayo_MSI_0006', 'Mayo_MSI_0007', 'Mayo_MSS_0012', 'Mayo_MSS_0014', 'Mayo_MSS_0002', 'Mayo_MSI_0015', 'Mayo_MSS_0019', 'Mayo_MSI_0012', 'Mayo_MSI_0004', 'Mayo_MSI_0010', 'Mayo_MSI_0019', 'Mayo_MSS_0013', 'Mayo_MSS_0006', 'Mayo_MSS_0008', 'Mayo_MSI_0014', 'Mayo_MSI_0002', 'Mayo_MSS_0009', 'Mayo_MSI_0005', 'Mayo_MSS_0005', 'Mayo_MSI_0020', 'Mayo_MSI_0009', 'Mayo_MSS_0017', 'Mayo_MSS_0016', 'Mayo_MSI_0017', 'Mayo_MSI_0016', 'Mayo_MSS_0018']


In [9]:
import skimage as ski
import os
all_image_pixels={}
all_image_slides = {}

#MSI Images
slide_dir = Path('/nfs/ml_lab/projects/Pilot1_PreclinicalHPC/mayo_data/Mayo_CRC_MSI_patches')
dirs = [x[0] for x in os.walk(slide_dir)]
for dir in dirs:
    if dir ==str(slide_dir):
        continue
    file = dir.split('/')[-1]
    slide_path = slide_dir / dir
    slides = os.listdir(slide_path)
    slides.sort()
    all_pixels={}
    all_slides={}
    patches=np.ndarray(shape=[len(slides), 224, 224, 3])
    i=0
    for slide in slides:
        slide1 = ski.io.imread(slide_path / slide)
        pixels = [int(slide.split('_')[4]),int(slide.split('_')[5])]
        all_pixels[slide] = pixels
        all_slides[slide] = slide1
        patches[i,:,:,:] = slide1
        i=i+1    
    all_image_pixels[file] = all_pixels
    all_image_slides[file] = all_slides

#MSS Images
slide_dir = Path('/nfs/ml_lab/projects/Pilot1_PreclinicalHPC/mayo_data/Mayo_CRC_MSS_patches')
dirs = [x[0] for x in os.walk(slide_dir)]
for dir in dirs:
    if dir ==str(slide_dir):
        continue
    file = dir.split('/')[-1]
    slide_path = slide_dir / dir
    slides = os.listdir(slide_path)
    slides.sort()
    all_pixels={}
    all_slides={}
    patches=np.ndarray(shape=[len(slides), 224, 224, 3])
    i=0
    for slide in slides:
        if (slide.split('-')[0] == 'Mayo_MSS_0017 '):
            continue
        slide1 = ski.io.imread(slide_path / slide)
        pixels = [int(slide.split('_')[4]),int(slide.split('_')[5])]
        all_pixels[slide] = pixels
        all_slides[slide] = slide1
        patches[i,:,:,:] = slide1
        i=i+1    
    all_image_pixels[file] = all_pixels
    all_image_slides[file] = all_slides


In [10]:
#Reconstruct image from patches
all_images = {}
#image_size=[1792, 672, 3]
image_size=[672, 1792, 3]
for im in all_image_slides.keys():
    if len(all_image_pixels[im].keys())!=24:  ### CHANGE THIS
        continue
    image = np.ndarray(shape=image_size)
    for key in all_image_slides[im].keys():
        p = all_image_pixels[im][key]
        image[p[1]-1:p[1]+223, p[0]-1:p[0]+223, :]=all_image_slides[im][key]
    all_images[im] = image    


In [11]:
all_images.keys()
savedir='./Results'
os.makedirs(savedir, exist_ok=True)

In [64]:
#Save_images
image_dir = (savedir+'/Original_images')
os.makedirs(image_dir, exist_ok=True)
for key in all_images.keys():
    #key = 'Mayo_MSI_0006_Tumor3'
    plt.figure()
    plt.imshow(all_images[key]/all_images[key].max())
    plt.title(key, fontsize=9)
    plt.savefig(str(image_dir+'/'+key+'.png'), pad_inches=0)
    plt.close()
    print(key)
    

Mayo_MSI_0006_TumorStroma3
Mayo_MSI_0017_TumorStroma1
Mayo_MSI_0007_TumorStroma3
Mayo_MSI_0007_Tumor3
Mayo_MSI_0015_TumorStroma1
Mayo_MSI_0009_Tumor1
Mayo_MSI_0012_Tumor1
Mayo_MSI_0004_Tumor1
Mayo_MSI_0011_Tumor3
Mayo_MSI_0014_TumorStroma1
Mayo_MSI_0001_TumorStroma3
Mayo_MSI_0011_TumorStroma1
Mayo_MSI_0010_TumorStroma1
Mayo_MSI_0012_TumorStroma1
Mayo_MSI_0010_Tumor1
Mayo_MSI_0002_TumorStroma3
Mayo_MSI_0006_Tumor1
Mayo_MSI_0008_TumorStroma3
Mayo_MSI_0009_TumorStroma3
Mayo_MSI_0017_TumorStroma2
Mayo_MSI_0015_Tumor3
Mayo_MSI_0009_Tumor2
Mayo_MSI_0015_TumorStroma2
Mayo_MSI_0014_TumorStroma2
Mayo_MSI_0011_TumorStroma2
Mayo_MSI_0010_TumorStroma2
Mayo_MSI_0002_Tumor1
Mayo_MSI_0017_Tumor3
Mayo_MSI_0010_Tumor2
Mayo_MSI_0012_TumorStroma2
Mayo_MSI_0001_Tumor3
Mayo_MSI_0006_Tumor2
Mayo_MSI_0014_Tumor1
Mayo_MSI_0017_Tumor1
Mayo_MSI_0002_Tumor3
Mayo_MSI_0002_TumorStroma2
Mayo_MSI_0014_Tumor3
Mayo_MSI_0001_Tumor1
Mayo_MSI_0001_TumorStroma2
Mayo_MSI_0007_Tumor2
Mayo_MSI_0015_Tumor1
Mayo_MSI_0011_Tumor

In [12]:
#Plot attention roll out maps
all_roll_out_map={}
image_size=[672,1792]
pixels_features

for im in rollout.keys():
    if len(all_image_pixels[im].keys())!=24:  ### CHANGE THIS
        continue
    roll_out_map = np.ndarray(shape=image_size)
    i=0
    pixels = pixels_features[im]
    for p in pixels:
        roll_out_map[p[1]-1:p[1]+223, p[0]-1:p[0]+223]=rollout[im][i]
        i=i+1
    roll_out_map=NormalizeData(roll_out_map)
    all_roll_out_map[im] = roll_out_map
    


In [29]:
#Save_images
image_dir = (savedir+'/Roll_out_maps')
os.makedirs(image_dir, exist_ok=True)
for key in all_roll_out_map.keys():
    plt.figure()
    plt.imshow(all_roll_out_map[key], cmap = 'viridis', vmin=0, vmax=1)
    plt.colorbar(orientation="horizontal")
    plt.title(key, fontsize=9)
    plt.savefig(str(image_dir+'/'+key+'.png'), pad_inches=0)
    plt.close()
    print(key)



Mayo_MSS_0005_Tumor1
Mayo_MSI_0002_TumorStroma1
Mayo_MSS_0005_Tumor2
Mayo_MSI_0002_TumorStroma2
Mayo_MSI_0002_TumorStroma3
Mayo_MSS_0005_Tumor3
Mayo_MSI_0007_Tumor3
Mayo_MSS_0019_Tumor1
Mayo_MSS_0005_TumorStroma2
Mayo_MSI_0002_Tumor1
Mayo_MSI_0007_TumorStroma1
Mayo_MSS_0005_TumorStroma3
Mayo_MSI_0007_Tumor2
Mayo_MSS_0019_Tumor2
Mayo_MSI_0002_Tumor3
Mayo_MSS_0005_TumorStroma1
Mayo_MSI_0007_TumorStroma3
Mayo_MSI_0007_TumorStroma2
Mayo_MSS_0019_Tumor3
Mayo_MSI_0007_Tumor1
Mayo_MSS_0019_TumorStroma3
Mayo_MSS_0019_TumorStroma2
Mayo_MSS_0019_TumorStroma1
Mayo_MSS_0002_Tumor3
Mayo_MSS_0002_Tumor2
Mayo_MSS_0002_Tumor1
Mayo_MSI_0011_Tumor2
Mayo_MSI_0011_TumorStroma2
Mayo_MSS_0002_TumorStroma2
Mayo_MSS_0002_TumorStroma3
Mayo_MSI_0011_Tumor3
Mayo_MSI_0011_Tumor1
Mayo_MSI_0011_TumorStroma1
Mayo_MSS_0002_TumorStroma1
Mayo_MSI_0008_TumorStroma2
Mayo_MSS_0018_Tumor1
Mayo_MSI_0008_TumorStroma3
Mayo_MSS_0018_Tumor2
Mayo_MSS_0018_Tumor3
Mayo_MSI_0008_Tumor1
Mayo_MSS_0018_TumorStroma1
Mayo_MSS_0018_Tumor

In [31]:
#Save_images
image_dir = (savedir+'/Roll_out_maps_with_images')
os.makedirs(image_dir, exist_ok=True)
for key in all_roll_out_map.keys():
    plt.figure()
    plt.imshow(all_images[key]/all_images[key].max())
    plt.imshow(all_roll_out_map[key], cmap = 'viridis', alpha=0.7)
    plt.colorbar(orientation="horizontal")
    plt.title(key, fontsize=9)
    plt.savefig(str(image_dir+'/'+key+'.png'), pad_inches=0)
    plt.close()
    print(key)



Mayo_MSS_0005_Tumor1
Mayo_MSI_0002_TumorStroma1
Mayo_MSS_0005_Tumor2
Mayo_MSI_0002_TumorStroma2
Mayo_MSI_0002_TumorStroma3
Mayo_MSS_0005_Tumor3
Mayo_MSI_0007_Tumor3
Mayo_MSS_0019_Tumor1
Mayo_MSS_0005_TumorStroma2
Mayo_MSI_0002_Tumor1
Mayo_MSI_0007_TumorStroma1
Mayo_MSS_0005_TumorStroma3
Mayo_MSI_0007_Tumor2
Mayo_MSS_0019_Tumor2
Mayo_MSI_0002_Tumor3
Mayo_MSS_0005_TumorStroma1
Mayo_MSI_0007_TumorStroma3
Mayo_MSI_0007_TumorStroma2
Mayo_MSS_0019_Tumor3
Mayo_MSI_0007_Tumor1
Mayo_MSS_0019_TumorStroma3
Mayo_MSS_0019_TumorStroma2
Mayo_MSS_0019_TumorStroma1
Mayo_MSS_0002_Tumor3
Mayo_MSS_0002_Tumor2
Mayo_MSS_0002_Tumor1
Mayo_MSI_0011_Tumor2
Mayo_MSI_0011_TumorStroma2
Mayo_MSS_0002_TumorStroma2
Mayo_MSS_0002_TumorStroma3
Mayo_MSI_0011_Tumor3
Mayo_MSI_0011_Tumor1
Mayo_MSI_0011_TumorStroma1
Mayo_MSS_0002_TumorStroma1
Mayo_MSI_0008_TumorStroma2
Mayo_MSS_0018_Tumor1
Mayo_MSI_0008_TumorStroma3
Mayo_MSS_0018_Tumor2
Mayo_MSS_0018_Tumor3
Mayo_MSI_0008_Tumor1
Mayo_MSS_0018_TumorStroma1
Mayo_MSS_0018_Tumor

In [35]:
## Trying to see how data is processed in histobistro durng model training
from data import MILDataset, get_multi_cohort_df
def get_data(cfg):
    data, clini_info = get_multi_cohort_df(
            cfg.data_config,
            cfg.cohorts, [cfg.target],
            cfg.label_dict,
            norm=cfg.norm,
            feats=cfg.feats,
            clini_info=cfg.clini_info
        )
    return data

train_dataset = MILDataset(
            get_data(cfg),
            train_idxs, [cfg.target],
            num_tiles=cfg.num_tiles,
            pad_tiles=cfg.pad_tiles,
            norm=cfg.norm,
            clini_info=cfg.clini_info
        )
#pd.read_excel("clini_table.xlsx", dtype=str)

NameError: name 'train_idxs' is not defined

In [47]:
data = get_data(cfg)
#data.index.name=None
data
#slide_path contains path to all 6 h5 embeddings of the patients.
#Line 266 in data.py within class MILDataset - Only the first h5 embedding of each patient is taken - Tumor 1

,PATIENT,TARGET,FILENAME,slide_path
0,Mayo_MSI_0001,1,Mayo_MSI_0001_Tumor1,"[h5_embeddings/Mayo_MSI_0001_Tumor1.h5, h5_emb..."
1,Mayo_MSI_0002,1,Mayo_MSI_0002_Tumor1,"[h5_embeddings/Mayo_MSI_0002_Tumor1.h5, h5_emb..."
2,Mayo_MSI_0003,1,Mayo_MSI_0003_Tumor1,"[h5_embeddings/Mayo_MSI_0003_Tumor1.h5, h5_emb..."
3,Mayo_MSI_0004,1,Mayo_MSI_0004_Tumor1,"[h5_embeddings/Mayo_MSI_0004_Tumor1.h5, h5_emb..."
4,Mayo_MSI_0005,1,Mayo_MSI_0005_Tumor1,"[h5_embeddings/Mayo_MSI_0005_Tumor1.h5, h5_emb..."
5,Mayo_MSI_0006,1,Mayo_MSI_0006_Tumor1,"[h5_embeddings/Mayo_MSI_0006_Tumor1.h5, h5_emb..."
6,Mayo_MSI_0007,1,Mayo_MSI_0007_Tumor1,"[h5_embeddings/Mayo_MSI_0007_Tumor1.h5, h5_emb..."
7,Mayo_MSI_0008,1,Mayo_MSI_0008_Tumor1,"[h5_embeddings/Mayo_MSI_0008_Tumor1.h5, h5_emb..."
8,Mayo_MSI_0009,1,Mayo_MSI_0009_Tumor1,"[h5_embeddings/Mayo_MSI_0009_Tumor1.h5, h5_emb..."
9,Mayo_MSI_0010,1,Mayo_MSI_0010_Tumor1,"[h5_embeddings/Mayo_MSI_0010_Tumor1.h5, h5_emb..."


In [52]:
f_name=str(file_name.split('_')[0]+'_'+file_name.split('_')[1]+'_'+file_name.split('_')[2])
id1=np.where(data['PATIENT']==f_name)[0]
data.iloc[id1]['TARGET'].values[0]

0

Compute Class Scores

In [58]:
import torchmetrics
name  = 'debug'
target = 'TARGET'
folds = 5
path = '/nfs/ml_lab/projects/Pilot1_PreclinicalHPC/mayo_data/HistoBistro/h5_embeddings'
files = glob.glob(str(path + '/*.h5'))
all_scores={}
all_acc={}
file_names=[]
for fold in range(folds):
    # Model
    model_path = Path(model_folds[fold])
    cfg.pos_weight = torch.tensor([1.0])
    classifier = ClassifierLightning(cfg)
    checkpoint = torch.load(model_path, map_location=lambda storage, loc: storage)
    checkpoint['state_dict'].keys()
    classifier.load_state_dict(checkpoint['state_dict'])
    classifier.eval();
    # Get roll out
    y=[]
    for file in file_folds[fold]:
        file_name = file.split('/')[-1].split('.')[0]
        file_names.append(file_name)
        f_name=str(file_name.split('_')[0]+'_'+file_name.split('_')[1]+'_'+file_name.split('_')[2])
        id1=np.where(data['PATIENT']==f_name)[0]
        y.append(data.iloc[id1]['TARGET'].values[0])
        h5_file = h5py.File(file)
        features = torch.Tensor(np.array(h5_file['features'])).unsqueeze(0)
        #features_all[file_name] = features
        n = features.shape[1]
        scores = np.zeros(n)
        for i in tqdm(range(n)):
            out = classifier.model(features[:, i:i+1, :]).squeeze(0)
            scores[i] = torch.sigmoid(out)
        acc = torchmetrics.Accuracy(task=cfg.task, num_classes=cfg.num_classes, multidim_average='samplewise')
        all_acc[file_name]=acc(scores, torch.tensor(y))
        all_scores[file_name] = scores
        


100%|██████████| 24/24 [00:00<00:00, 223.94it/s]


RuntimeError: Predictions and targets are expected to have the same shape, but got (24,) and torch.Size([1]).

In [30]:
data.iloc[0]['slide_path']

[PosixPath('h5_embeddings/Mayo_MSI_0001_Tumor1.h5'),
 PosixPath('h5_embeddings/Mayo_MSI_0001_Tumor2.h5'),
 PosixPath('h5_embeddings/Mayo_MSI_0001_Tumor3.h5'),
 PosixPath('h5_embeddings/Mayo_MSI_0001_TumorStroma1.h5'),
 PosixPath('h5_embeddings/Mayo_MSI_0001_TumorStroma2.h5'),
 PosixPath('h5_embeddings/Mayo_MSI_0001_TumorStroma3.h5')]

In [14]:
#Plot scores maps
all_scores_map={}
image_size=[672,1792]

for im in all_scores.keys():
    if len(all_image_pixels[im].keys())!=24:  ### CHANGE THIS
        continue
    scores_map = np.ndarray(shape=image_size)
    i=0
    pixels = pixels_features[im]
    for p in pixels:
        scores_map[p[1]-1:p[1]+223, p[0]-1:p[0]+223]=all_scores[im][i]
        i=i+1
    #scores_map=NormalizeData(scores_map)
    all_scores_map[im] = scores_map
    

In [32]:
#Save_images
image_dir = (savedir+'/Classification_maps')
os.makedirs(image_dir, exist_ok=True)
for key in all_scores_map.keys():
    plt.figure()
    plt.imshow(all_scores_map[key], vmin=0, vmax=1, cmap = 'RdBu')
    colorbar=plt.colorbar(orientation="horizontal")
    colorbar.set_ticks([0, 0.5, 1])
    plt.title(key, fontsize=9)
    plt.savefig(str(image_dir+'/'+key+'.png'), pad_inches=0)
    plt.close()
    print(key)


Mayo_MSS_0005_Tumor1
Mayo_MSI_0002_TumorStroma1
Mayo_MSS_0005_Tumor2
Mayo_MSI_0002_TumorStroma2
Mayo_MSI_0002_TumorStroma3
Mayo_MSS_0005_Tumor3
Mayo_MSI_0007_Tumor3
Mayo_MSS_0019_Tumor1
Mayo_MSS_0005_TumorStroma2
Mayo_MSI_0002_Tumor1
Mayo_MSI_0007_TumorStroma1
Mayo_MSS_0005_TumorStroma3
Mayo_MSI_0007_Tumor2
Mayo_MSS_0019_Tumor2
Mayo_MSI_0002_Tumor3
Mayo_MSS_0005_TumorStroma1
Mayo_MSI_0007_TumorStroma3
Mayo_MSI_0007_TumorStroma2
Mayo_MSS_0019_Tumor3
Mayo_MSI_0007_Tumor1
Mayo_MSS_0019_TumorStroma3
Mayo_MSS_0019_TumorStroma2
Mayo_MSS_0019_TumorStroma1
Mayo_MSS_0002_Tumor3
Mayo_MSS_0002_Tumor2
Mayo_MSS_0002_Tumor1
Mayo_MSI_0011_Tumor2
Mayo_MSI_0011_TumorStroma2
Mayo_MSS_0002_TumorStroma2
Mayo_MSS_0002_TumorStroma3
Mayo_MSI_0011_Tumor3
Mayo_MSI_0011_Tumor1
Mayo_MSI_0011_TumorStroma1
Mayo_MSS_0002_TumorStroma1
Mayo_MSI_0008_TumorStroma2
Mayo_MSS_0018_Tumor1
Mayo_MSI_0008_TumorStroma3
Mayo_MSS_0018_Tumor2
Mayo_MSS_0018_Tumor3
Mayo_MSI_0008_Tumor1
Mayo_MSS_0018_TumorStroma1
Mayo_MSS_0018_Tumor

Image with classification scores

In [33]:
#Save_images
image_dir = (savedir+'/Class_maps_with_images')
os.makedirs(image_dir, exist_ok=True)
for key in all_scores_map.keys():
    plt.figure()
    plt.imshow(all_images[key]/all_images[key].max())
    plt.imshow(all_scores_map[key], vmin=0, vmax=1, cmap = 'RdBu', alpha=0.7)
    colorbar=plt.colorbar(orientation="horizontal")
    colorbar.set_ticks([0, 0.5, 1])
    plt.title(key, fontsize=9)
    plt.savefig(str(image_dir+'/'+key+'.png'), pad_inches=0)
    plt.close()
    print(key)


Mayo_MSS_0005_Tumor1
Mayo_MSI_0002_TumorStroma1
Mayo_MSS_0005_Tumor2
Mayo_MSI_0002_TumorStroma2
Mayo_MSI_0002_TumorStroma3
Mayo_MSS_0005_Tumor3
Mayo_MSI_0007_Tumor3
Mayo_MSS_0019_Tumor1
Mayo_MSS_0005_TumorStroma2
Mayo_MSI_0002_Tumor1
Mayo_MSI_0007_TumorStroma1
Mayo_MSS_0005_TumorStroma3
Mayo_MSI_0007_Tumor2
Mayo_MSS_0019_Tumor2
Mayo_MSI_0002_Tumor3
Mayo_MSS_0005_TumorStroma1
Mayo_MSI_0007_TumorStroma3
Mayo_MSI_0007_TumorStroma2
Mayo_MSS_0019_Tumor3
Mayo_MSI_0007_Tumor1
Mayo_MSS_0019_TumorStroma3
Mayo_MSS_0019_TumorStroma2
Mayo_MSS_0019_TumorStroma1
Mayo_MSS_0002_Tumor3
Mayo_MSS_0002_Tumor2
Mayo_MSS_0002_Tumor1
Mayo_MSI_0011_Tumor2
Mayo_MSI_0011_TumorStroma2
Mayo_MSS_0002_TumorStroma2
Mayo_MSS_0002_TumorStroma3
Mayo_MSI_0011_Tumor3
Mayo_MSI_0011_Tumor1
Mayo_MSI_0011_TumorStroma1
Mayo_MSS_0002_TumorStroma1
Mayo_MSI_0008_TumorStroma2
Mayo_MSS_0018_Tumor1
Mayo_MSI_0008_TumorStroma3
Mayo_MSS_0018_Tumor2
Mayo_MSS_0018_Tumor3
Mayo_MSI_0008_Tumor1
Mayo_MSS_0018_TumorStroma1
Mayo_MSS_0018_Tumor

In [15]:
# Plot original image, rollout maps and classification maps together

image_dir = (savedir+'/Original_attention_class_maps')
os.makedirs(image_dir, exist_ok=True)
for key in all_images.keys():
    fig, axs = plt.subplots(3)
    im=axs[0].imshow(all_images[key]/all_images[key].max())
    #plt.colorbar(im, ax=axs[0])
    axs[0].set_title('Original slide', fontsize=12)

    axs[1].imshow(all_images[key]/all_images[key].max())
    im=axs[1].imshow(all_roll_out_map[key], cmap = 'viridis', alpha=0.7)
    divider = make_axes_locatable(axs[1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)
    axs[1].set_title('Attention roll-out map', fontsize=12)

    axs[2].imshow(all_images[key]/all_images[key].max())
    im=axs[2].imshow(all_scores_map[key], vmin=0, vmax=1, cmap = 'RdBu', alpha=0.7)
    divider = make_axes_locatable(axs[2])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    colorbar=plt.colorbar(im, cax=cax)
    axs[2].set_title('Classification score map', fontsize=12)
    plt.subplots_adjust(left=None, bottom=0.1, right=None, top=1.3, wspace=0.5, hspace=None)
    plt.suptitle(key, fontsize=12)
    plt.tight_layout()
    plt.savefig(str(image_dir+'/'+key+'.png'), pad_inches=0)
    plt.close()    
    